# 👑 CRACKIE – THE LIVING MIRROR
**„Was wäre, wenn dein Spiegelbild plötzlich lebendig wird?“** – 120 s Voxel-Kurzfilm, 1080×1920, 24 fps.

**Bedienung vom iPhone:** oben *Laufzeit → Laufzeittyp ändern → T4-GPU* wählen, dann *Laufzeit → Alle ausführen*.
Alles wird in **Google Drive → `MyDrive/Crackie`** gespeichert. Bricht Colab ab, einfach erneut *Alle ausführen* –
fertige Schritte und Frames werden übersprungen (Checkpoints).

Nur kostenlose Werkzeuge: Blender (GPL), Piper TTS + Thorsten-Stimmen (frei, kein Voice-Cloning),
selbst synthetisierte Musik/Geräusche (numpy), Schriften unter SIL Open Font License, ffmpeg.

## 1 · GPU prüfen

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Keine GPU – es wird auf der CPU gerendert (sehr langsam)."

## 2 · Google Drive verbinden

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/Crackie'
for d in ['', 'audio/voice', 'voices', 'fonts', 'frames']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
print('Projektordner:', PROJ)

## 3 · Code holen und Werkzeuge installieren

In [ ]:
%cd /content
!rm -rf crackie_repo && git clone -q --depth 1 https://github.com/Dlschad20k-creator/test crackie_repo
SRC = '/content/crackie_repo/crackie/src'
!pip -q install piper-tts fonttools brotli scipy pillow numpy
BL_VER = '5.2.3'
BL = f'/content/blender-{BL_VER}-linux-x64/blender'
if not os.path.exists(BL):
    !wget -q https://download.blender.org/release/Blender5.2/blender-{BL_VER}-linux-x64.tar.xz -O /content/blender.tar.xz
    !tar -xf /content/blender.tar.xz -C /content && rm /content/blender.tar.xz
!{BL} --version | head -1
!ffmpeg -version | head -1

## 4 · Deutsche Stimmen (Piper, offline, kein Voice-Cloning)

In [ ]:
HF = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/de/de_DE'
for path in ['thorsten/high/de_DE-thorsten-high', 'thorsten_emotional/medium/de_DE-thorsten_emotional-medium']:
    name = path.split('/')[-1]
    for ext in ['.onnx', '.onnx.json']:
        dst = f'{PROJ}/voices/{name}{ext}'
        if not os.path.exists(dst):
            !wget -q {HF}/{path}{ext} -O "{dst}"
if not os.path.exists(f'{PROJ}/audio/voice/durations.json'):
    !cd {SRC} && python tts.py "{PROJ}/voices" "{PROJ}/audio/voice"
else:
    print('Stimmen schon vorhanden – übersprungen.')

## 5 · Schriften, Musik und Sounddesign (selbst synthetisiert)

In [ ]:
!cd {SRC} && python fonts.py "{PROJ}/fonts"
if not os.path.exists(f'{PROJ}/audio/mix.wav'):
    !cd {SRC} && python sound.py "{PROJ}/audio/voice" "{PROJ}/audio"
else:
    print('Tonmischung schon vorhanden – übersprungen.')

## 6 · Szene bauen (Voxel-Modelle, Bad, Spiegelwelt, Animation, Kamera, Licht)

In [ ]:
BLEND = f'{PROJ}/crackie.blend'
if not os.path.exists(BLEND):
    !cd {SRC} && {BL} -b --factory-startup --python build.py -- "{BLEND}" "{PROJ}/audio/voice" 2>&1 | grep -E "Keyframes|gespeichert|Error"
else:
    print('Szene schon gebaut – übersprungen. (Zum Neubauen crackie.blend in Drive löschen.)')

## 7 · Rendern – jede Szene einzeln, mit Checkpoints
Automatische GPU-Erkennung (OptiX/CUDA). Mit GPU: native 1080×1920. Ohne GPU: 540×960 + Hochskalierung.
Fertige Frames liegen in Drive und werden bei einem Neustart übersprungen.

In [ ]:
from IPython.display import clear_output
for s in range(1, 9):
    print(f'=== Szene {s} ===')
    !cd {SRC} && {BL} -b --factory-startup --python render.py -- "{BLEND}" "{PROJ}/frames" --scene {s} --quality auto 2>&1 | grep -E "GPU|CPU-Rendering|FERTIG|Error"
n = len([f for f in os.listdir(f'{PROJ}/frames') if f.endswith('.png') and 'tmp' not in f])
print(f'{n} / 2880 Frames fertig')

## 8 · Nachbearbeitung + Export (Untertitel, Titel, Glitch, Zensur, Endkarte) → MP4 H.264/AAC

In [ ]:
OUT = f'{PROJ}/Crackie_The_Living_Mirror.mp4'
!cd {SRC} && python post.py "{PROJ}/frames" "{PROJ}/crackie_meta.json" "{PROJ}/fonts" "{PROJ}/audio/mix.wav" "{OUT}"

## 9 · Qualitätskontrolle

In [ ]:
!cd {SRC} && python qc.py "{OUT}" "{PROJ}/frames" "{PROJ}/audio/mix.wav" "{PROJ}/fonts" "{PROJ}/qc_report.json"

## 10 · Ansehen

In [ ]:
from IPython.display import HTML
from base64 import b64encode
print('Datei in Google Drive:', OUT, f'({os.path.getsize(OUT)/1e6:.1f} MB)')
HTML(f'<video width=300 controls playsinline src="data:video/mp4;base64,{b64encode(open(OUT,"rb").read()).decode()}"></video>')